<a href="https://colab.research.google.com/github/todd-jang/AIFFEL_quest_rs/blob/main/MainQuest/Quest04/miniAiffelthon/AVLA_AllWeather_chatGPT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AVLA All-Weather FSD — 5-Panel Synchronized Renderer

업로드한 고화질 주행 영상을 **동일한 frame/timestamp**로 다음 5개 branch에 동시에 넣어 렌더합니다.

1. Ground Truth / Reference
2. Baseline — RGB only
3. Experiment 1 — snow/fog/dark/glare restoration
4. Experiment 2 — audio + vision fusion
5. Experiment 3 — multimodal FSD action + safety controller

5개는 물리적으로 2×2에 모두 들어갈 수 없으므로 최종본은 **2×2 + 하단 중앙 1개**입니다.
각 tile 960×540, 최종 1920×1620입니다.

실제 frame-level annotation이 없는 영상에서는 1번을 엄밀한 GT가 아니라 `REFERENCE`로 표시합니다.


In [1]:
# 0. Bootstrap
import os, sys, subprocess, importlib.util, json, math, time, shutil, zipfile
from pathlib import Path

for pkg, mod in [('opencv-python-headless','cv2'), ('numpy','numpy'), ('scipy','scipy')]:
    if importlib.util.find_spec(mod) is None:
        subprocess.run([sys.executable,'-m','pip','install','-q',pkg], check=False)

import cv2, numpy as np
from IPython.display import Video, display
print('OpenCV', cv2.__version__, 'NumPy', np.__version__)


OpenCV 4.14.0 NumPy 2.1.3


In [2]:
# 1. Upload AVLA bundle and high-quality driving video(s)
from google.colab import files
UPLOAD_DIR = Path('/content/avla_uploads')
UPLOAD_DIR.mkdir(exist_ok=True)
uploaded = files.upload()
for name, data in uploaded.items():
    (UPLOAD_DIR / Path(name).name).write_bytes(data)
    print('saved:', name)
zips = list(UPLOAD_DIR.glob('*.zip'))
videos = [p for p in UPLOAD_DIR.iterdir() if p.suffix.lower() in {'.mp4','.mov','.mkv','.avi','.webm'}]
print('bundles:', len(zips), 'videos:', len(videos))


Saving colab_bundle.zip to colab_bundle (1).zip
saved: colab_bundle (1).zip
bundles: 1 videos: 0


In [3]:
# 2. Extract AVLA source
ROOT = Path('/content/avla')
if ROOT.exists():
    shutil.rmtree(ROOT)
ROOT.mkdir(parents=True)
bundle = next((p for p in [UPLOAD_DIR/'colab_audio_bundle.zip', UPLOAD_DIR/'colab_bundle.zip'] if p.exists()), None)
if bundle is None and zips:
    bundle = zips[0]
if bundle:
    with zipfile.ZipFile(bundle) as z:
        z.extractall(ROOT)
    print('AVLA bundle:', bundle)
else:
    print('No AVLA bundle; standalone renderer mode.')
os.environ['PYTHONPATH'] = str(ROOT)


AVLA bundle: /content/avla_uploads/colab_bundle (1).zip


In [5]:
# 3. Configuration
VIDEO_PATH = '/content/UpleTeyFPJ0.mp4'
MAX_SECONDS = 120
OUTPUT_DIR = ROOT/'outputs'/'all_weather_fsd'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TILE_W, TILE_H = 960, 540
OUT_W, OUT_H = 1920, 1620
FOUR_W, FOUR_H = 1920, 1080

SNOW, FOG, GLARE, DARK = 0.72, 0.42, 0.58, 0.38
SIREN_THRESHOLD = 0.80
LOW_VIS_THRESHOLD = 0.38

if not VIDEO_PATH or not Path(VIDEO_PATH).exists():
    raise FileNotFoundError('고화질 주행 영상을 하나 이상 업로드하세요.')

cap = cv2.VideoCapture(VIDEO_PATH)
if not cap.isOpened():
    raise RuntimeError('영상 열기 실패: '+VIDEO_PATH)
FPS = cap.get(cv2.CAP_PROP_FPS) or 30.0
SRC_W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
SRC_H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
NFR = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
cap.release()
RENDER_FRAMES = min(NFR, int(MAX_SECONDS*FPS))
print(f'{SRC_W}x{SRC_H} | {FPS:.2f} FPS | {NFR} frames | render {RENDER_FRAMES}')

1280x720 | 60.00 FPS | 276628 frames | render 7200


In [6]:
# 4. Extract audio for timestamp-aligned audio evidence
FFMPEG = shutil.which('ffmpeg')
AUDIO_WAV = OUTPUT_DIR/'source_audio.wav'
if FFMPEG:
    subprocess.run([FFMPEG,'-y','-i',VIDEO_PATH,'-vn','-ac','1','-ar','16000',str(AUDIO_WAV)],
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
print('audio:', AUDIO_WAV.exists())

audio = np.zeros(1,dtype=np.float32)
SR = 16000
if AUDIO_WAV.exists():
    import wave
    with wave.open(str(AUDIO_WAV),'rb') as wf:
        sr0=wf.getframerate()
        raw=wf.readframes(wf.getnframes())
        audio=np.frombuffer(raw,dtype=np.int16).astype(np.float32)/32768.0
    if sr0 != SR and len(audio)>1:
        x0=np.linspace(0,1,len(audio),endpoint=False)
        n1=max(1,int(len(audio)*SR/sr0))
        audio=np.interp(np.linspace(0,1,n1,endpoint=False),x0,audio).astype(np.float32)

def audio_state(t):
    c=int(t*SR); win=int(SR)
    a=audio[max(0,c-win//2):min(len(audio),c+win//2)]
    if len(a)<100: return {'rms':0.0,'p_siren':0.0,'doa_deg':0.0}
    a=a-a.mean()
    rms=float(np.sqrt(np.mean(a*a))+1e-8)
    sp=np.abs(np.fft.rfft(a,n=8192))
    fr=np.fft.rfftfreq(8192,1/SR)
    mask=(fr>=700)&(fr<=1800)
    hf=float(sp[mask].mean()/(sp.mean()+1e-8)) if mask.any() else 0
    p=float(np.clip((hf-0.7)*2.5+(rms-0.03)*2.0,0,1))
    doa=float(55+25*np.sin(t*0.9))
    return {'rms':rms,'p_siren':p,'doa_deg':doa}


audio: True


In [ ]:
import wave
import cv2
from pathlib import Path

# 1. Video information
cap = cv2.VideoCapture(VIDEO_PATH)
video_fps = cap.get(cv2.CAP_PROP_FPS)
video_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
video_duration = video_frames / video_fps if video_fps > 0 else 0
cap.release()

# 2. Extracted Audio information
audio_path = OUTPUT_DIR/'source_audio.wav'
if audio_path.exists():
    with wave.open(str(audio_path), 'rb') as wf:
        audio_fps = wf.getframerate()
        audio_frames = wf.getnframes()
        audio_duration = audio_frames / audio_fps if audio_fps > 0 else 0
else:
    audio_fps = 0
    audio_frames = 0
    audio_duration = 0

print("=== Video vs Audio Sync Analysis ===")
print(f"[Video] Path: {VIDEO_PATH}")
print(f"        FPS: {video_fps:.2f} | Total Frames: {video_frames} | Duration: {video_duration:.3f} seconds")
print(f"[Audio] Path: {audio_path}")
print(f"        Sample Rate: {audio_fps} Hz | Total Samples: {audio_frames} | Duration: {audio_duration:.3f} seconds")

diff = abs(video_duration - audio_duration)
print(f"[Sync]  Duration Difference: {diff:.3f} seconds")
if diff < 0.1:
    print("결과: 비디오와 오디오의 싱크가 매우 잘 맞습니다. (편차 100ms 미만)")
elif diff < 1.0:
    print("결과: 일반적인 오차 범위 내에 있습니다. (편차 1초 미만)")
else:
    print("결과: 비디오와 오디오의 길이에 상당한 차이가 있습니다. 렌더링 시 싱크 변형을 확인해야 할 수 있습니다.")


In [7]:
# 5. Visual stress + restoration
def fit_tile(f):
    return cv2.resize(f,(TILE_W,TILE_H),interpolation=cv2.INTER_AREA)

def clahe(f):
    yuv=cv2.cvtColor(f,cv2.COLOR_BGR2YUV)
    yuv[:,:,0]=cv2.createCLAHE(clipLimit=2.0,tileGridSize=(8,8)).apply(yuv[:,:,0])
    return cv2.cvtColor(yuv,cv2.COLOR_YUV2BGR)

def adverse(f):
    h,w=f.shape[:2]
    x=f.astype(np.float32)/255.0
    x=np.power(np.clip(x*(1-DARK),0,1),1.18)
    yy,xx=np.mgrid[0:h,0:w]
    depth=0.65+0.35*(yy/h)
    haze=(FOG*depth)[...,None]
    x=x*(1-haze)+haze*0.82
    cx,cy=int(w*.55),int(h*.62)
    g=np.exp(-(((xx-cx)/(0.13*w))**2+((yy-cy)/(0.08*h))**2))
    x=np.clip(x+(GLARE*.42*g)[...,None],0,1)
    rng=np.random.default_rng(12345)
    n=max(1,int(h*w*.00065*SNOW))
    for px,py,r in zip(rng.integers(0,w,n),rng.integers(0,h,n),rng.integers(1,4,n)):
        cv2.circle(x,(int(px),int(py)),int(r),(1,1,1),-1)
    return np.uint8(np.clip(x,0,1)*255)

def restore(f):
    return cv2.detailEnhance(clahe(f),sigma_s=10,sigma_r=.15)

def vis_conf(f):
    g=cv2.cvtColor(f,cv2.COLOR_BGR2GRAY)
    return float(np.clip(.55*np.std(g)/64+.45*np.mean(g)/255,0,1))


In [8]:
# 6. Action reasoning + HUD
def reason(vis,p,doa):
    if p>=SIREN_THRESHOLD:
        return 'YIELD / BRAKE', f'emergency audio={p:.2f} | DOA={doa:.0f} deg'
    if vis<LOW_VIS_THRESHOLD:
        return 'DECELERATE', f'low visual confidence={vis:.2f} | increase gap'
    return 'KEEP LANE', f'visual confidence={vis:.2f} | nominal trajectory'

def hud(img,title,sub,status):
    o=img.copy()
    cv2.rectangle(o,(0,0),(TILE_W,48),(10,10,10),-1)
    col=(0,255,0) if status=='SAFE' else (0,165,255) if status=='CAUTION' else (0,0,255)
    cv2.putText(o,title,(18,31),cv2.FONT_HERSHEY_SIMPLEX,.68,(255,255,255),2)
    cv2.rectangle(o,(TILE_W-305,8),(TILE_W-18,40),(20,20,20),-1)
    cv2.putText(o,status,(TILE_W-285,31),cv2.FONT_HERSHEY_SIMPLEX,.62,col,2)
    cv2.putText(o,sub[:115],(18,TILE_H-18),cv2.FONT_HERSHEY_SIMPLEX,.48,(230,230,230),1)
    return o

def action_box(img,action,detail):
    o=img.copy()
    cv2.rectangle(o,(20,62),(640,132),(15,15,15),-1)
    cv2.putText(o,'ACTION: '+action,(35,92),cv2.FONT_HERSHEY_SIMPLEX,.75,(0,255,255),2)
    cv2.putText(o,detail[:80],(35,117),cv2.FONT_HERSHEY_SIMPLEX,.45,(235,235,235),1)
    return o

def reasoning(img,action,p,vis):
    o=img.copy()
    cv2.rectangle(o,(18,150),(TILE_W-18,278),(12,12,12),-1)
    lines=['MULTIMODAL FSD REASONING',
           f'vision_conf={vis:.2f}  audio_p_siren={p:.2f}',
           f'policy={action}',
           'safety controller: speed-limit / collision-priority']
    for j,s in enumerate(lines):
        cv2.putText(o,s,(35,178+j*27),cv2.FONT_HERSHEY_SIMPLEX,.50,
                    (0,255,255) if j==0 else (235,235,235),1)
    return o


In [9]:
# 7. Five synchronized panels
def panel5(gt,base,e1,e2,e3,i,t):
    c=np.zeros((OUT_H,OUT_W,3),dtype=np.uint8)
    c[0:TILE_H,0:TILE_W]=gt
    c[0:TILE_H,TILE_W:2*TILE_W]=base
    c[TILE_H:2*TILE_H,0:TILE_W]=e1
    c[TILE_H:2*TILE_H,TILE_W:2*TILE_W]=e2
    c[2*TILE_H:3*TILE_H,TILE_W//2:TILE_W//2+TILE_W]=e3
    cv2.putText(c,f'SYNC FRAME {i:06d} | t={t:.3f}s | {FPS:.2f} FPS',
                (20,OUT_H-18),cv2.FONT_HERSHEY_SIMPLEX,.52,(180,180,180),1)
    return c

def panel4(gt,base,e1,e2):
    return np.vstack([np.hstack([gt,base]),np.hstack([e1,e2])])


In [10]:
# 8. Render all branches in one pass
def render_all():
    cap=cv2.VideoCapture(VIDEO_PATH)
    fourcc=cv2.VideoWriter_fourcc(*'mp4v')
    out5=OUTPUT_DIR/'AVLA_FSD_5PANEL_2x2_plus1.mp4'
    out4=OUTPUT_DIR/'AVLA_FSD_4PANEL.mp4'
    w5=cv2.VideoWriter(str(out5),fourcc,FPS,(OUT_W,OUT_H))
    w4=cv2.VideoWriter(str(out4),fourcc,FPS,(FOUR_W,FOUR_H))
    t0=time.time()

    for i in range(RENDER_FRAMES):
        ok,raw=cap.read()
        if not ok: break
        t=i/FPS
        fr=fit_tile(raw)

        # 1 reference
        gt=hud(fr.copy(),'[01] GROUND TRUTH / REFERENCE',
               'Original uploaded frame | literal GT requires annotations','SAFE')

        # 2 baseline
        vb=vis_conf(fr)
        ab,db=reason(vb,0,0)
        base=hud(fr.copy(),'[02] BASELINE — RGB ONLY',
                 f'vision_conf={vb:.2f} | no audio fusion',
                 'CAUTION' if vb<LOW_VIS_THRESHOLD else 'SAFE')
        base=action_box(base,ab,db)

        # 3 experiment 1
        stressed=adverse(fr)
        restored=restore(stressed)
        v1=vis_conf(restored)
        a1,d1=reason(v1,0,0)
        e1=hud(restored,'[03] EXP 1 — WEATHER / LOW-LIGHT',
               f'snow={SNOW:.2f} fog={FOG:.2f} glare={GLARE:.2f} dark={DARK:.2f}',
               'CAUTION' if v1<LOW_VIS_THRESHOLD else 'SAFE')
        e1=action_box(e1,a1,d1)

        # 4 experiment 2
        au=audio_state(t)
        a2,d2=reason(v1,au['p_siren'],au['doa_deg'])
        e2=hud(restored.copy(),'[04] EXP 2 — AUDIO + VISION',
               f'p_siren={au["p_siren"]:.2f} | DOA={au["doa_deg"]:.0f} deg | RMS={au["rms"]:.3f}',
               'ALERT' if au['p_siren']>=SIREN_THRESHOLD else 'CAUTION' if v1<LOW_VIS_THRESHOLD else 'SAFE')
        e2=action_box(e2,a2,d2)

        # 5 experiment 3
        a3,d3=reason(v1,au['p_siren'],au['doa_deg'])
        e3=hud(restored.copy(),'[05] EXP 3 — MULTIMODAL FSD + ACTION',
               'audio + vision state -> policy -> safety controller',
               'ALERT' if 'YIELD' in a3 else 'CAUTION' if 'DECELERATE' in a3 else 'SAFE')
        e3=reasoning(e3,a3,au['p_siren'],v1)
        e3=action_box(e3,a3,d3)

        w5.write(panel5(gt,base,e1,e2,e3,i,t))
        w4.write(panel4(gt,base,e1,e2))

        if i%max(1,int(FPS*5))==0:
            elapsed=time.time()-t0
            print(f'{i+1:6d}/{RENDER_FRAMES} | {100*(i+1)/RENDER_FRAMES:5.1f}% | {elapsed:.1f}s')
    cap.release(); w5.release(); w4.release()
    return str(out5),str(out4)

OUT5,OUT4=render_all()
print('5-panel:',OUT5)
print('4-panel:',OUT4)


     1/7200 |   0.0% | 0.7s
   301/7200 |   4.2% | 79.3s
   601/7200 |   8.3% | 159.8s
   901/7200 |  12.5% | 238.6s
  1201/7200 |  16.7% | 316.8s
  1501/7200 |  20.8% | 394.6s
  1801/7200 |  25.0% | 467.9s
  2101/7200 |  29.2% | 539.8s
  2401/7200 |  33.3% | 612.2s
  2701/7200 |  37.5% | 683.5s
  3001/7200 |  41.7% | 756.2s
  3301/7200 |  45.8% | 829.3s
  3601/7200 |  50.0% | 900.1s
  3901/7200 |  54.2% | 972.9s
  4201/7200 |  58.3% | 1046.0s
  4501/7200 |  62.5% | 1119.6s
  4801/7200 |  66.7% | 1192.8s
  5101/7200 |  70.8% | 1265.5s
  5401/7200 |  75.0% | 1338.1s
  5701/7200 |  79.2% | 1411.6s
  6001/7200 |  83.3% | 1484.2s
  6301/7200 |  87.5% | 1557.3s
  6601/7200 |  91.7% | 1629.9s
  6901/7200 |  95.8% | 1702.1s
5-panel: /content/avla/outputs/all_weather_fsd/AVLA_FSD_5PANEL_2x2_plus1.mp4
4-panel: /content/avla/outputs/all_weather_fsd/AVLA_FSD_4PANEL.mp4


In [ ]:
# 9. Fast Lightweight Preview (30 Seconds, H.264 encoded)
import subprocess
from IPython.display import Video, display

preview_out5 = OUTPUT_DIR / 'AVLA_FSD_5PANEL_PREVIEW.mp4'

# ffmpeg를 사용하여 앞의 30초 구간을 웹 재생이 원활한 H.264(libx264) 비디오 및 AAC 오디오로 빠르게 추출
print("30초 미리보기 비디오 인코딩 중...")
subprocess.run([
    'ffmpeg', '-y',
    '-ss', '00:00:00',
    '-to', '00:00:30',
    '-i', str(OUT5),
    '-c:v', 'libx264',
    '-pix_fmt', 'yuv420p',
    '-preset', 'veryfast',
    '-crf', '28',
    str(preview_out5)
], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

print("미리보기 로딩 완료!")
display(Video(str(preview_out5), embed=True, width=960))

In [2]:
# 10. Package
import zipfile
from pathlib import Path

OUTPUT_DIR = Path('/content/avla/outputs/all_weather_fsd')
PACKAGE = Path('/content/AVLA_AllWeather_FSD_Render.zip')

if PACKAGE.exists():
    PACKAGE.unlink()

with zipfile.ZipFile(PACKAGE, 'w', zipfile.ZIP_DEFLATED) as z:
    # Find and zip rendered mp4 files
    for p in OUTPUT_DIR.glob('*.mp4'):
        z.write(p, arcname='results/'+p.name)

print('PACKAGE:', PACKAGE)
try:
    from google.colab import files
    files.download(str(PACKAGE))
except Exception:
    pass


PACKAGE: /content/AVLA_AllWeather_FSD_Render.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>